# DBZD Phase 0 Kaggle runner
Edit only the input paths and Kaggle Dataset slug below. This runner completes the seven missing phase0_final_r3 runs.

In [ ]:
REPO_URL = "https://github.com/Alessio2405/DBZD.git"
GITHUB_SECRET_NAME = "GITHUB_TOKEN"  # optional for public repositories
RUN_MATRIX = [
    ("multitask", 44),
    ("dbzd_full", 42), ("dbzd_full", 43), ("dbzd_full", 44),
    ("dbzd_stopgrad", 42), ("dbzd_stopgrad", 43), ("dbzd_stopgrad", 44),
]
CONFIG = "configs/default.yaml"
EXPERIMENT_REVISION = "phase0_final_r3"
DATAGEN_SCHEMA_VERSION = 3
RUNS_INPUT_DIR = ""  # e.g. /kaggle/input/dbzd-phase0-runs/runs
RUNS_INPUT_ZIP = ""  # e.g. /kaggle/input/dbzd-phase0-runs/runs.zip
KAGGLE_DATASET_SLUG = "YOUR_USERNAME/dbzd-phase0-runs"
DATASET_TITLE = "DBZD Phase 0 runs"
RUN_PROBES = True
AUTO_FIX_P100 = True


In [ ]:
import json, os, shutil, subprocess
from pathlib import Path

work = Path("/kaggle/working")
repo = work / "dbzd"
if repo.exists():
    shutil.rmtree(repo)

clone_url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(GITHUB_SECRET_NAME)
except Exception:
    token = None
if token and clone_url.startswith("https://"):
    clone_url = clone_url.replace("https://", f"https://x-access-token:{token}@", 1)
subprocess.run(["git", "clone", "--quiet", clone_url, str(repo)], check=True)
os.chdir(repo)
print("Repository ready at", repo)

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
compat_command = ["python", "scripts/kaggle_torch_compat.py"]
if AUTO_FIX_P100:
    compat_command.append("--auto-fix-p100")
subprocess.run(compat_command, check=True)

In [ ]:
runs = repo / "runs"
if RUNS_INPUT_ZIP:
    restored = work / "restored-runs"
    if restored.exists():
        shutil.rmtree(restored)
    shutil.unpack_archive(RUNS_INPUT_ZIP, restored)
    source = restored / "runs"
    if not source.exists():
        source = restored
    shutil.copytree(source, runs, dirs_exist_ok=True)
    print("Restored runs from", RUNS_INPUT_ZIP)
elif RUNS_INPUT_DIR:
    source = Path(RUNS_INPUT_DIR)
    if source.exists():
        shutil.copytree(source, runs, dirs_exist_ok=True)
        print("Restored runs from", source)
runs.mkdir(exist_ok=True)

data_dir = repo / "data" / "phase0"
metadata_path = data_dir / "metadata.json"
data_revision = None
if metadata_path.exists():
    data_revision = json.loads(metadata_path.read_text()).get("generator_schema_version")
if data_revision != DATAGEN_SCHEMA_VERSION:
    if data_dir.exists():
        shutil.rmtree(data_dir)
    print("Generating datagen schema", DATAGEN_SCHEMA_VERSION)
    subprocess.run([
        "python", "-m", "datagen", "--output-dir", str(data_dir),
        "--tokenizer", "HuggingFaceTB/SmolLM-135M",
        "--train-n", "40000", "--val-n", "2000", "--test-n", "2000"
    ], check=True)

In [ ]:
import csv, math, yaml
def validate_alpha_run(run_dir):
    summary = json.loads((run_dir / "summary.json").read_text())
    alpha_init = float((summary.get("config") or {}).get("alpha_init", 0.3))
    with (run_dir / "metrics.csv").open() as handle:
        rows = [row for row in csv.DictReader(handle) if row.get("split") == "val"]
    trajectory = [(int(row["global_step"]), float(row["alpha"]), float(row["alpha_lm_gradient"])) for row in rows]
    print("dbzd_full alpha trajectory:", trajectory)
    if not trajectory or any(not math.isfinite(gradient) for _, _, gradient in trajectory):
        raise RuntimeError(f"dbzd_full alpha_lm_gradient is missing/non-finite: {trajectory}")
    if not any(abs(alpha - alpha_init) > 1e-7 for _, alpha, _ in trajectory):
        raise RuntimeError(f"dbzd_full alpha never moved from {alpha_init}: {trajectory}")

for arm, seed in RUN_MATRIX:
    run_dir = runs / f"{arm}_s{seed}"
    resolved = run_dir / "resolved_config.yaml"
    summary_path = run_dir / "summary.json"
    old_revision = None
    completed_summary = False
    if run_dir.exists():
        old_summary = None
        if summary_path.exists():
            old_summary = json.loads(summary_path.read_text())
            completed_summary = bool(old_summary.get("completed"))
        if resolved.exists():
            old_revision = (yaml.safe_load(resolved.read_text()) or {}).get("experiment_revision")
        elif old_summary is not None:
            old_revision = (old_summary.get("config") or {}).get("experiment_revision")
        if old_revision != EXPERIMENT_REVISION:
            raise RuntimeError(f"Refusing to delete {run_dir}: revision {old_revision!r} != {EXPERIMENT_REVISION!r}")
    if (run_dir / "model_final.pt").exists() or completed_summary:
        print("Skipping completed", arm, seed)
        if RUN_PROBES and not (run_dir / "probe_summary.json").exists():
            if (run_dir / "model_final.pt").exists():
                subprocess.run(["python", "probe.py", "--run-dir", str(run_dir)], check=True)
            else:
                print("WARNING: probe missing and no model checkpoint for", run_dir)
        continue
    command = [
        "python", "train.py", "--config", CONFIG,
        "--data-dir", str(data_dir), "--run-root", str(runs),
        "--arm", arm, "--seed", str(seed)
    ]
    if (run_dir / "checkpoint_latest.pt").exists():
        command.append("--resume")
    print("Running", arm, seed, "resume=" + str("--resume" in command))
    subprocess.run(command, check=True)
    if arm == "dbzd_full":
        validate_alpha_run(run_dir)
    if RUN_PROBES:
        subprocess.run(["python", "probe.py", "--run-dir", str(run_dir)], check=True)

subprocess.run(["python", "analysis.py", "--runs-dir", str(runs)], check=True)

In [ ]:
import json

export = work / "dbzd-runs-export"
if export.exists():
    shutil.rmtree(export)
export.mkdir()
export_runs = export / "runs"
export_runs.mkdir()
artifact_names = {"metrics.csv", "summary.json", "probe_summary.json"}
for run_dir in sorted(runs.glob("*_s*")):
    if not run_dir.is_dir():
        continue
    target = export_runs / run_dir.name
    target.mkdir()
    for name in artifact_names:
        source = run_dir / name
        if source.exists():
            shutil.copy2(source, target / name)
if (runs / "analysis").exists():
    shutil.copytree(runs / "analysis", export_runs / "analysis")
metadata = {
    "title": DATASET_TITLE,
    "id": KAGGLE_DATASET_SLUG,
    "licenses": [{"name": "CC0-1.0"}]
}
(export / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))
if "YOUR_USERNAME" not in KAGGLE_DATASET_SLUG:
    subprocess.run([
        "kaggle", "datasets", "version", "-p", str(export),
        "-m", "DBZD Phase 0 checkpoint and metrics update", "--dir-mode", "zip"
    ], check=True)
else:
    print("Set KAGGLE_DATASET_SLUG to publish a Dataset version.")
shutil.make_archive(str(work / "dbzd-runs"), "zip", root_dir=export_runs)
print("Packaged", work / "dbzd-runs.zip")